# Dataloader, Patient-Level Splitting & Augmentation
This notebook implements the data pipeline using `nibabel` and defines the `BraTS3DDataset`. It ensures strict patient-level splitting (80/20) and handles multi-class target engineering (WT, TC, ET) from BraTS 2023 masks.


In [1]:
import os
import glob
import torch
import numpy as np
import nibabel as nib
from torch.utils.data import Dataset, DataLoader
import pandas as pd

# Constants
DATA_DIR = "../../dataset/brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
PATCH_SIZE = (96, 96, 96)

class BraTS3DDataset(Dataset):
    def __init__(self, patient_dirs, patch_size=PATCH_SIZE, transform=None):
        self.patient_dirs = patient_dirs
        self.patch_size = patch_size
        self.transform = transform

    def __len__(self):
        return len(self.patient_dirs)

    def __getitem__(self, idx):
        patient_dir = self.patient_dirs[idx]
        
        # Paths
        t1n_path = glob.glob(os.path.join(patient_dir, "*-t1n.nii*"))[0]
        t1c_path = glob.glob(os.path.join(patient_dir, "*-t1c.nii*"))[0]
        t2w_path = glob.glob(os.path.join(patient_dir, "*-t2w.nii*"))[0]
        t2f_path = glob.glob(os.path.join(patient_dir, "*-t2f.nii*"))[0]
        seg_path = glob.glob(os.path.join(patient_dir, "*-seg.nii*"))[0]
        
        # Load NIfTI as numpy arrays
        # Shape is (H, W, D) in nibabel. We'll transpose later to (C, D, H, W).
        t1n = nib.load(t1n_path).get_fdata().astype(np.float32)
        t1c = nib.load(t1c_path).get_fdata().astype(np.float32)
        t2w = nib.load(t2w_path).get_fdata().astype(np.float32)
        t2f = nib.load(t2f_path).get_fdata().astype(np.float32)
        seg = nib.load(seg_path).get_fdata().astype(np.float32)
        
        # Stack inputs along channel axis -> (4, H, W, D)
        image_4d = np.stack([t1n, t1c, t2w, t2f], axis=0) 
        
        # Move Depth to front for (C, D, H, W)
        image_4d = np.transpose(image_4d, (0, 3, 1, 2))
        seg_3d = np.transpose(seg, (2, 0, 1)) # (D, H, W)
                
        # Padding if necessary
        d, h, w = image_4d.shape[1:]
        pd, ph, pw = self.patch_size
        
        if d < pd or h < ph or w < pw:
            image_4d = np.pad(image_4d, ((0,0), (0, max(0, pd-d)), (0, max(0, ph-h)), (0, max(0, pw-w))))
            seg_3d = np.pad(seg_3d, ((0, max(0, pd-d)), (0, max(0, ph-h)), (0, max(0, pw-w))))
            d, h, w = image_4d.shape[1:]

        # Random 3D Crop
        z = np.random.randint(0, d - pd + 1)
        y = np.random.randint(0, h - ph + 1)
        x = np.random.randint(0, w - pw + 1)
        
        image_patch = image_4d[:, z:z+pd, y:y+ph, x:x+pw]
        seg_patch = seg_3d[z:z+pd, y:y+ph, x:x+pw]
        
        # Target Engineering: BraTS 2023 Mask to Clinical Channels
        # NCR (1), ED (2), ET (3)
        # WT = 1+2+3 (Any tumor presence)
        # TC = 1+3 (Core + Enhancing, exclude Edema)
        # ET = 3
        
        wt_mask = (seg_patch > 0).astype(np.float32)
        tc_mask = np.logical_or(seg_patch == 1, seg_patch == 3).astype(np.float32)
        et_mask = (seg_patch == 3).astype(np.float32)
        
        mask_channels = np.stack([wt_mask, tc_mask, et_mask], axis=0) # (3, D, H, W)
        
        sample = {
            'image': torch.from_numpy(image_patch),
            'mask': torch.from_numpy(mask_channels)
        }
        
        if self.transform:
            sample = self.transform(sample)
            
        return sample


In [2]:
# Patient-Level Splitting and Loading Example
all_patients = [os.path.join(DATA_DIR, d) for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
np.random.shuffle(all_patients)

split_idx = int(0.8 * len(all_patients))
train_patients = all_patients[:split_idx]
val_patients = all_patients[split_idx:]

print(f"Total Patients: {len(all_patients)}")
print(f"Train Patients: {len(train_patients)}")
print(f"Val Patients: {len(val_patients)}")

if len(all_patients) > 0:
    train_dataset = BraTS3DDataset(patient_dirs=train_patients)
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
    
    # Test fetch a batch
    batch = next(iter(train_loader))
    print(f"Image tensor shape: {batch['image'].shape}") # Expected: (1, 4, 96, 96, 96)
    print(f"Mask tensor shape: {batch['mask'].shape}")   # Expected: (1, 3, 96, 96, 96)


Total Patients: 1251
Train Patients: 1000
Val Patients: 251
Image tensor shape: torch.Size([1, 4, 96, 96, 96])
Mask tensor shape: torch.Size([1, 3, 96, 96, 96])
